In [142]:
import os
import pandas as pd
import requests
from dotenv import load_dotenv
import csv

In [143]:
load_dotenv()
base_url = "https://api.insee.fr/api-sirene/3.11/siret/"
sirene_api_key = os.environ.get("SIRENE_API_KEY")

In [144]:
# open codes NAF
naf_codes = []
with open("output/interesting_naf_codes.csv", "r", encoding="utf-8") as file:
    csv_reader = csv.DictReader(file, delimiter="|")
    for line in csv_reader:
        naf_codes.append(line["Code NAF"][:2] + "." + line["Code NAF"][2:])

In [145]:
print(naf_codes)

['35.11Z', '35.12Z', '35.13Z', '35.14Z', '35.21Z', '35.22Z', '35.23Z', '36.00Z', '37.00Z', '38.11Z', '38.12Z', '38.21Z', '38.22Z', '38.32Z', '39.00Z', '32.11Z', '32.12Z', '32.13Z', '32.20Z', '32.30Z', '32.40Z', '32.50A', '32.50B', '32.91Z', '32.99Z', '33.11Z', '33.12Z', '33.13Z', '33.14Z', '33.15Z', '33.16Z', '33.17Z', '33.19Z', '33.20A', '33.20B', '33.20C', '33.20D', '24.10Z', '24.20Z', '24.31Z', '24.32Z', '24.33Z', '24.34Z', '24.42Z', '24.42Z', '24.43Z', '24.44Z', '24.45Z', '24.46Z', '24.51Z', '24.52Z', '24.53Z', '24.54Z', '25.11Z', '25.12Z', '25.21Z', '25.29Z', '25.30Z', '25.40Z', '25.50A', '25.50B', '25.61Z', '25.62A', '25.62B', '25.71Z', '25.72Z', '25.73A', '25.73B', '25.91Z', '25.92Z', '25.93Z', '25.94Z', '25.99A', '25.99B', '21.10Z', '21.20Z', '28.11Z', '28.12Z', '28.13Z', '28.14Z', '28.15Z', '28.21Z', '28.22Z', '28.23Z', '28.24Z', '28.25Z', '28.29A', '28.29B', '28.30Z', '28.41Z', '28.49Z', '28.91Z', '28.92Z', '28.93Z', '28.94Z', '28.95Z', '28.96Z', '28.99A', '28.99B', '20.11Z',

In [146]:
naf_codes = pd.read_csv("output/interesting_naf_codes.csv", delimiter='|')['Code NAF']

In [147]:
naf_codes = naf_codes.apply(lambda x: x[:2] + "." + x[2:]).unique()

In [148]:
headers = {
    "X-INSEE-Api-Key-Integration": sirene_api_key,
    "Accept-Encoding": "gzip",
    "Accept": "application/json",
}

In [149]:
departement = "69"

In [150]:
from tqdm import tqdm

champs = "siret,activitePrincipaleUniteLegale,trancheEffectifsUniteLegale,codeCommuneEtablissement,coordonneeLambertAbscisseEtablissement"

for code in tqdm(naf_codes):
    query = f"activitePrincipaleUniteLegale:{code} AND trancheEffectifsUniteLegale:[0 TO 53] AND codeCommuneEtablissement:{departement}*"
    # Parameters for the query
    params = {
        "q": query,
        "champs": champs,
        "nombre": "1000"
    }
    response = requests.get(
        url=base_url,
        headers=headers,
        params=params
    )
    try:
        if response.json()["header"]["total"] > 1000:
            print(f"plus de 1000 entrées sur : {query}")
    except Exception as e:
        print(e)
        print(response.json())
    break

  0%|          | 0/276 [00:00<?, ?it/s]


In [151]:
from pandas.io.json._normalize import json_normalize

In [152]:
etablissements = json_normalize(response.json()['etablissements'])

In [153]:
response.json()["header"]

{'statut': 200, 'message': 'OK', 'total': 390, 'debut': 0, 'nombre': 390}

In [154]:
etablissements.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 390 entries, 0 to 389
Data columns (total 5 columns):
 #   Column                                                       Non-Null Count  Dtype 
---  ------                                                       --------------  ----- 
 0   siret                                                        390 non-null    object
 1   uniteLegale.activitePrincipaleUniteLegale                    390 non-null    object
 2   uniteLegale.trancheEffectifsUniteLegale                      390 non-null    object
 3   adresseEtablissement.codeCommuneEtablissement                390 non-null    object
 4   adresseEtablissement.coordonneeLambertAbscisseEtablissement  324 non-null    object
dtypes: object(5)
memory usage: 15.4+ KB


# slicing

In [155]:
from tqdm import tqdm

champs = "siret,activitePrincipaleUniteLegale,trancheEffectifsUniteLegale,codeCommuneEtablissement,coordonneeLambertAbscisseEtablissement,coordonneeLambertOrdonneeEtablissement"

taille_slice = 20

etablissements = pd.DataFrame()

for i in tqdm(range(0, len(naf_codes), taille_slice)):
    naf_groupe = naf_codes[i:min(i+taille_slice, len(naf_codes)-1)]
    query = f"trancheEffectifsUniteLegale:[0 TO 53] AND codeCommuneEtablissement:{departement}* AND (activitePrincipaleUniteLegale:{naf_groupe[0]} "
    for code in naf_groupe[1:]:
        query += f"OR activitePrincipaleUniteLegale:{code} "
    query += ")"
    # Parameters for the query
    params = {
        "q": query,
        "champs": champs,
        "nombre": "1000"
    }
    response = requests.get(
        url=base_url,
        headers=headers,
        params=params
    )
    try:
        if response.json()["header"]["total"] > 1000:
            print(f"plus de 1000 entrées sur : {query}")
        # executer le traitement normal
        etablissements = pd.concat([etablissements, json_normalize(response.json()['etablissements'])])
    except Exception as e:
        print(e)
        print(response.json())

 50%|█████     | 7/14 [00:01<00:01,  3.57it/s]

plus de 1000 entrées sur : trancheEffectifsUniteLegale:[0 TO 53] AND codeCommuneEtablissement:62* AND (activitePrincipaleUniteLegale:10.31Z OR activitePrincipaleUniteLegale:10.32Z OR activitePrincipaleUniteLegale:10.39A OR activitePrincipaleUniteLegale:10.39B OR activitePrincipaleUniteLegale:10.41A OR activitePrincipaleUniteLegale:10.41B OR activitePrincipaleUniteLegale:10.42Z OR activitePrincipaleUniteLegale:10.51A OR activitePrincipaleUniteLegale:10.51B OR activitePrincipaleUniteLegale:10.51C OR activitePrincipaleUniteLegale:10.51D OR activitePrincipaleUniteLegale:10.52Z OR activitePrincipaleUniteLegale:10.61A OR activitePrincipaleUniteLegale:10.61B OR activitePrincipaleUniteLegale:10.62Z OR activitePrincipaleUniteLegale:10.71A OR activitePrincipaleUniteLegale:10.71B OR activitePrincipaleUniteLegale:10.71C OR activitePrincipaleUniteLegale:10.71D OR activitePrincipaleUniteLegale:10.72Z )


100%|██████████| 14/14 [00:03<00:00,  4.25it/s]


In [156]:
# etablissements = json_normalize(response.json()['etablissements'])

In [157]:
etablissements["adresseEtablissement.coordonneeLambertAbscisseEtablissement"].value_counts()

adresseEtablissement.coordonneeLambertAbscisseEtablissement
603875.6296829311    16
700326.713543167     14
684258.3687486494    12
[ND]                 12
695220.9619558346    11
                     ..
697395.3930483996     1
663755.5323365051     1
639723.1276027939     1
681770.6540099679     1
702322.7524930923     1
Name: count, Length: 2950, dtype: int64

In [158]:
etablissements["adresseEtablissement.coordonneeLambertOrdonneeEtablissement"].value_counts()

adresseEtablissement.coordonneeLambertOrdonneeEtablissement
7070434.947677597     16
7036333.57430482      14
7022151.922632079     12
[ND]                  12
7040903.751027016     11
                      ..
7034167.3207572065     1
7028360.889737204      1
7053425.855074266      1
7022544.120433443      1
7001258.757001699      1
Name: count, Length: 2950, dtype: int64

In [159]:
etablissements["adresseEtablissement.coordonneeLambertOrdonneeEtablissement"] = pd.to_numeric(etablissements["adresseEtablissement.coordonneeLambertOrdonneeEtablissement"], errors='coerce')
etablissements["adresseEtablissement.coordonneeLambertAbscisseEtablissement"] = pd.to_numeric(etablissements["adresseEtablissement.coordonneeLambertAbscisseEtablissement"], errors='coerce')

In [160]:
etablissements.dropna(inplace=True)

In [161]:
# verify that we are able to plot 
import geopandas as gpd

In [162]:
geo_etablissement = gpd.GeoDataFrame(
    etablissements,
    geometry=gpd.points_from_xy(etablissements["adresseEtablissement.coordonneeLambertAbscisseEtablissement"], etablissements["adresseEtablissement.coordonneeLambertOrdonneeEtablissement"]),
    crs="EPSG:9794"
    )

In [163]:
geo_etablissement = geo_etablissement.to_crs("EPSG:3857")

In [164]:
geo_etablissement['x'] = geo_etablissement.geometry.x
geo_etablissement['y'] = geo_etablissement.geometry.y

In [165]:
from bokeh.io import output_file, output_notebook, show
from bokeh.plotting import figure, ColumnDataSource
output_notebook()

Loading BokehJS ...

In [166]:
# Step 2: Convert to ColumnDataSource for Bokeh
source = ColumnDataSource(data=dict(
    x=geo_etablissement['x'],
    y=geo_etablissement['y'],
    # add more columns here if you want tooltips or coloring based on other attributes
))

In [167]:
# Step 3: Create the Bokeh plot
p = figure(title="Geospatial Points", x_axis_type="mercator", y_axis_type="mercator")
p.add_tile("OSM")  # Add OpenStreetMap tiles (or other tile provider)

# Plot the points
p.circle(x='x', y='y', source=source, size=5, color="blue", alpha=0.7)

# Display the plot
show(p)